In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException
import time
from utils import writeJson, readJson
import os
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
import json
import re
from bs4 import BeautifulSoup as soup
import datetime
from datetime import datetime as dt
from tqdm import tqdm

In [ ]:
def getPlayerStats(driver):
    data = {}
    tables = driver.find_elements(By.CLASS_NAME, 'stats_table')
    table = None
    for t in tables:
        if "scout_full" in t.get_attribute("id"):
            table = t
            break
    tags = ['<br>', '<strong>', '</strong>']
    type_stat = 'Standard Stats'
    data[type_stat] = {}
    for row in table.find_elements(By.TAG_NAME, 'tr'):
        th = row.find_element(By.TAG_NAME, 'th')
        #print(f'--{row.get_property("className")} -- {row.text}')
        if row.get_property("className") == "thead over_header thead":
            type_stat = th.text
            data[type_stat] = {}
            #print(type_stat)
        
        data_desc = th.get_attribute('data-tip')
        

        tds = row.find_elements(By.TAG_NAME, 'td')
        if len(tds) > 0 and th.text != '':
            value, perc = tds[0].text , tds[1].text
            #f'{th.text} ({data_desc})'
            for t in tags:
                if data_desc != None:
                    data_desc = data_desc.replace(t, ' ')
                
            data[type_stat][th.text] = {'description': data_desc, 'value': value, 'percentile': perc.strip()}

    return data

def getRoles(role):
    rr=[]
    if '(' in role:
        roles_split = role.split('(')
    else:
        roles_split = [role]

    for r in roles_split:
        if '-' in r:
            r_split = r.split('-')
            rr.append(r_split[0])
            rl = r_split[1]
            if rl[-1] == ')':
                rl = rl[:-1]
            rr.append(rl)

        elif ',' in r:
            rr.append(r.split(',')[0])
        elif ')' in r:
            rr.append(r[:-1])
        else:
            rr.append(r.strip())


    return rr

def initializeDriver():
    driver = webdriver.Chrome()
    url = 'https://fbref.com/en/'
    driver.get(url)
    cookie_button = driver.find_elements(By.TAG_NAME, 'button')
    for b in cookie_button:
        if b.text == 'Accetta tutto':
            b.click()
    return driver

def getPlayerAnag(driver, player_dict):
    try:
        more_button = driver.find_element(By.XPATH,'//*[@id="meta_more_button"]')
        more_button.click()
    except:
        pass

    #anag_div = driver.find_element(By.XPATH, '/html/body/div[4]/div[3]/div[1]/div[2]')
    #player = anag_div.find_element(By.TAG_NAME, 'h1').text
    i=1
    anag_elem1= ''

    try:
        driver.find_element(By.XPATH,f'//*[@id="meta"]/div[2]')
        tab_prefix = '//*[@id="meta"]/div[2]'
    except:
        tab_prefix = '//*[@id="meta"]/div'
    
    while not anag_elem1.startswith("Position"):
        anag_elem1 = driver.find_element(By.XPATH,f'{tab_prefix}/p[{i}]').text
        i+=1
        if i==10:
            raise KeyError
    #anag_elem1 = driver.find_element(By.XPATH,'//*[@id="meta"]/div[2]/p[1]').text
    anag_elem2 = driver.find_element(By.XPATH,f'{tab_prefix}/p[{i}]').text
    anag_elem3 = driver.find_element(By.XPATH,f'{tab_prefix}/p[{i+1}]').text
    #anag_elem4 = driver.find_element(By.XPATH,f'{tab_prefix}/p[{i+2}]').text
    '''
    try:
        anag_elem5 = driver.find_element(By.XPATH,f'//*[@id="meta"]/div[2]/p[{i+3}]').text
    except:
        anag_elem5 = ''
    '''
    anag_elem1_split = anag_elem1.split('▪')
    #anag_elem2_split = anag_elem2.split(' ')
    position = anag_elem1_split[0].split(':')[1].strip()
    #footed = anag_elem1_split[1].split(':')[1].strip()
    if anag_elem2.startswith('Born'):
        year_birth = anag_elem2.split(' ')[3]
        #height = ''
        #weight = ''
        nat = anag_elem3.split(' ')[2] if anag_elem3.startswith('National Team') else anag_elem3.split(' ')[1]
        #team = ' '.join(anag_elem5.split(' ')[1:]) if anag_elem4 != '' else ''

    else:
        #height, weight = anag_elem2_split[0].replace(',',''), anag_elem2_split[1]
        year_birth = anag_elem3.split(' ')[3]
        #nat = anag_elem4.split(' ')[2] if anag_elem3.startswith('National Team') else anag_elem3.split(' ')[1]
        #team = ' '.join(anag_elem5.split(' ')[1:]) if anag_elem5 != '' else ''
        
    addict = dict(#player=player, 
                position=position 
                #foot=footed, 
                #height= height, 
                #weight=weight, 
                ,year_birth=year_birth
                #,nationality=nat
                #,team=team
                )

    return player_dict | addict


def getPlayerRecord(driver, player_dict):
    url = player_dict['link']
    driver.get(url)
    time.sleep(0.5)
    try:
        stats = getPlayerStats(driver)
        player_dict['stats'] = stats
    except:
        pass
    if 'stats' in player_dict.keys():
        player_dict = getPlayerAnag(driver, player_dict)
    else:
        player_dict['stats'] = {}
    
    
    return player_dict


def getTeamPlayers(driver, url : str, id_league: int, team: str) -> list:
    urls=[]
    driver.get(url)
    #id_league = '12229'
    #table = driver.find_element(By.XPATH, '//*[@id="stats_standard_11"]/tbody')
    table = driver.find_element(By.CLASS_NAME, 'stats_table')
    
    rows = table.find_elements(By.TAG_NAME, 'tr')
    for r in rows:
        presenze = '0'
        th = r.find_element(By.TAG_NAME, 'th')
        if th.get_attribute('csk') != None:
            tds = r.find_elements(By.TAG_NAME, 'td')
            for td in tds:
                if td.get_attribute('data-stat') == 'games':
                    presenze = td.text
            if int(presenze) > 5:
                player_name = th.find_element(By.TAG_NAME, 'a').get_attribute('href').split('/')[-1].replace('-',' ')
                #player_name = th.text
                id_player = th.get_attribute('data-append-csv')
                #print(th.text,th.get_attribute('data-append-csv'), presenze)
                url_p=f"https://fbref.com/en/players/{id_player}/scout/{id_league}/{player_name.replace(' ', '-')}-Scouting-Report"
                urls.append(dict(id=id_player, name=player_name, link =url_p, team=team))
    return urls

def getLeagueTeams(driver, url, id):
    driver.get(url)
    team_links=[]
    #table = driver.find_element(By.XPATH, '//*[@id="results2023-202491_overall"]/tbody')
    #table = driver.find_element(By.TAG_NAME, 'tbody')
    table = driver.find_element(By.CLASS_NAME, 'stats_table')
    table = table.find_element(By.TAG_NAME, 'tbody')
    #print(table.text)
    rows = table.find_elements(By.TAG_NAME, 'tr')
    for r in rows:
        l = r.find_element(By.TAG_NAME,'a')
        team = l.text
        link= l.get_property('href')
        team_links.append(dict(id=id, team=team, link=link))
    return team_links


In [4]:
prompts = {}
records = readJson('Dataset/Fbref/prova_records_perc_v3.json')
for p in records:
    player_name = p['player']
    prompt = f""""You are a professional football scout with expertise in analyzing players' technical and tactical characteristics. 
            I need you to generate a detailed report for a player, based on the provided list of statistics that describe their performance averaged per 90 minutes. 
            For each statistics, it is indicate value and percentile. Percentile is a value between 0 and 100. High value for percentile means that the player is good in that statistic.
            Percentile comparison is made between players of same role.
            Your task is to analyze this data and provide a report as follows:

            ### Input Data:
                - Player: {player_name}
                - Position: {p['position']}
                - Year birth: {p['year_birth']}
                - Height: {p['height']}
                - Weight: {p['weight']}
                - Statistics per 90 minutes: 
                {p['stats']}

            ### Output Format:
            Your report should be structured in the following way:
            **Player**: {player_name}
            **Strengths**: 
            Highlight the player's key strengths evident from their playing style.
            **Weaknesses**: 
            Point out areas where the player needs improvement.
            **Summary**:
            A brief summary of the player's overall performance.


            ### Notes for Analysis:
            - Use concise and professional language.
            - The report should be realistic for scouting purposes.
            - Do not generate code or class structures. Focus only on the football analysis.
            - The output must be in plain text, clearly formatted according to the structure above.
            - Do not write the name of the player 
            - Do not include statistics into report
            
            ###Generated Report:"""
    prompts[player_name] = {'prompt': prompt}

writeJson(prompts, 'Descriptions/prova_stats.json')

Il file non è stato trovato.


TypeError: 'NoneType' object is not iterable

In [27]:
records = readJson('Dataset/Fbref/players.json')
records = [x for x in records if 'position' in x.keys() and x['position'] != 'GK']
len(records)

1831

Dati link leghe estrae i link di tutte le squadre

In [ ]:
leagues = readJson('Dataset/Fbref/competitions.json')
team_leagues = []
driver=initializeDriver()
for id, link in leagues.items():
    print(id,link)
    teams = getLeagueTeams(driver, link, id)
    team_leagues= team_leagues + teams

writeJson(team_leagues, 'Dataset/Fbref/teams.json')

Scrittura giocatori per ogni squadra

In [485]:
team_leagues = readJson('Dataset/Fbref/teams.json')
player_links= []
driver=initializeDriver()
with tqdm(total=len(team_leagues), desc="Extracting players link") as pbar:
    for team in team_leagues:
        players_team = getTeamPlayers(driver, team['link'], team['id'], team['team'])
        player_links= player_links + players_team
        #print(player_links)
        pbar.update(1)

driver.quit()
writeJson(player_links, 'Dataset/Fbref/players.json')

Extracting players link: 100%|██████████| 96/96 [45:11<00:00, 28.25s/it]


Estrae statistiche per ogni giocatore presente nel file di input

In [4]:
players = readJson('Dataset/Fbref/players.json')
driver = initializeDriver()
with tqdm(total=len(players), desc="Processing players") as pbar:

    for i in range(len(players)):
        if 'stats' not in players[i].keys():
            players[i] = getPlayerRecord(driver, players[i])
        
        if i %10 == 0:
            writeJson(players, 'Dataset/Fbref/players.json')
        
        pbar.update(1)
writeJson(players, 'Dataset/Fbref/players.json')
driver.quit()

Processing players: 100%|██████████| 2309/2309 [45:11<00:00,  1.17s/it]


Nicolo Zaniolo {'id': '9907b1c8', 'name': 'Nicolo Zaniolo', 'link': 'https://fbref.com/en/players/9907b1c8/scout/12192/Nicolo-Zaniolo-Scouting-Report', 'team': 'Aston Villa', 'stats': {'Standard Stats': {'Goals': {'description': 'Goals scored or allowed', 'value': '0.21', 'percentile': '47'}, 'Assists': {'description': 'Assists', 'value': '0.00', 'percentile': '2'}, 'Goals + Assists': {'description': 'Goals and Assists', 'value': '0.21', 'percentile': '12'}, 'Non-Penalty Goals': {'description': 'Non-Penalty Goals', 'value': '0.21', 'percentile': '51'}, 'Penalty Kicks Made': {'description': 'Penalty Kicks Made', 'value': '0.00', 'percentile': '37'}, 'Penalty Kicks Attempted': {'description': 'Penalty Kicks Attempted', 'value': '0.00', 'percentile': '36'}, 'Yellow Cards': {'description': 'Yellow Cards', 'value': '0.75', 'percentile': '1'}, 'Red Cards': {'description': 'Red Cards', 'value': '0.00', 'percentile': '54'}, 'xG: Expected Goals': {'description': 'Expected Goals xG totals includ

In [186]:
dataset = readJson('Dataset/transfermarkt_fbref_dataset.json')
dataset_stat_null = [x for x in dataset if x['stats'] == {}]
for i in range(len(dataset_stat_null)):
    dataset_stat_null[i]['elab'] = 0
writeJson(dataset_stat_null, 'Dataset/transfermarkt_fbref_mancanti.json')

In [187]:
dataset_stat_null = readJson('Dataset/transfermarkt_fbref_mancanti.json')
#driver = initializeDriver()
driver = initializeDriver()
with tqdm(total=len(dataset_stat_null), desc="Processing players") as pbar:
    for i in range(len(dataset_stat_null)):
        
        if dataset_stat_null[i]['elab'] == 0:
            player_stat = getPlayerRecord(driver,dataset_stat_null[i])
            dataset_stat_null[i] = player_stat
            dataset_stat_null[i]['elab'] = 1
            writeJson(dataset_stat_null, 'Dataset/transfermarkt_fbref_mancanti.json')

        pbar.update(1)

driver.quit()

Processing players:   0%|          | 0/353 [00:00<?, ?it/s]


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"xpath","selector":"//*[@id="meta"]/div/p[1]"}
  (Session info: chrome=133.0.6943.142); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x00007FF72297C6A5+28789]
	(No symbol) [0x00007FF7228E5B20]
	(No symbol) [0x00007FF722778F9A]
	(No symbol) [0x00007FF7227CF346]
	(No symbol) [0x00007FF7227CF57C]
	(No symbol) [0x00007FF722822B17]
	(No symbol) [0x00007FF7227F736F]
	(No symbol) [0x00007FF72281F7E3]
	(No symbol) [0x00007FF7227F7103]
	(No symbol) [0x00007FF7227BFFC0]
	(No symbol) [0x00007FF7227C1273]
	GetHandleVerifier [0x00007FF722CC1AED+3458237]
	GetHandleVerifier [0x00007FF722CD829C+3550316]
	GetHandleVerifier [0x00007FF722CCDB9D+3507565]
	GetHandleVerifier [0x00007FF722A42C6A+841274]
	(No symbol) [0x00007FF7228F09EF]
	(No symbol) [0x00007FF7228ECB34]
	(No symbol) [0x00007FF7

Prompt per generare descrizione giocatori

In [140]:
prompts = {}
#records = readJson('Dataset/Fbref/players.json')
records = readJson('Dataset/transfermarkt_fbref_dataset.json')
#records = [x for x in records if x['team'] in ['Roma','Milan','Juventus','Inter','Atalanta','Real Madrid', 'Manchester City']]
#records = [x for x in records if x['team'] in ['Milan']]
records = [x for x in records if 'position' in x.keys() and x['position'] != 'GK']
#records = [x for x in records if x['name'] in ['Alessandro Bastoni', 'Theo Hernandez', 'Benjamin Pavard', 'Fikayo Tomori', 'Francesco Acerbi']]

position_dict = readJson('Dataset/Fbref/position_mapping.json')
#stats_dict = {'Offensive':['Shooting', 'Goal and Shot Creation', 'Possession','Passing', 'Pass Types'], 'Defensive':['Defense', 'Miscellaneous Stats']}
stats_dict = {'Shooting':['Shooting'], 'Passing': ['Goal and Shot Creation','Passing', 'Pass Types'], 'Possession': ['Possession'], 'Defensive':['Defense']}#, 'Miscellaneous Stats']}
for p in records:
    player_name = p['id']#p['name']

    #print(player_name, p['name'], p['link'], p['position'])
    prompts[player_name] = {'prompt':{}}
    roles = getRoles(p['position'])
    roles_verb = [position_dict[x.strip()] for x in roles]
    roles_str = ', '.join(roles_verb)
    #stats_dict = p['stats']
    prompts[player_name]['position_str'] = roles_str
    prompts[player_name]['name'] = p['name']
    for k, tab  in list(stats_dict.items())[0:]:
        stat = tab
        stat = {}
        for t in tab:
            stat = stat | p['stats'][t]

        prompt = f""""You are a professional soccer scout with expertise in analyzing players' technical and tactical characteristics.
        ## Task:  
        Generate a very concise description (75 tokens) of a player's {k} performance based on the provided per-match statistics. 
        The analysis should:    
            - Highlight the player's **key strengths** (high percentiles).
            - Identify potential **areas for improvement** (low percentiles).
            - Be **role-specific**, considering the player's position.
        
        ## Reasoning Process:
            - For each statistic, analyze its percentile ranking, classifying performance using these thresholds:
                Excellent (≥90th percentile) -> Major strength
                Very Good (75-89th percentile) -> Significant asset
                Good (50-74th percentile) -> Competent ability
                Average (30-49th percentile) -> Room for improvement
                Weak (<30th percentile) -> Notable weakness

            - Interpret the player's style of play:
                Identify how the player's strengths shape their contributions.
                Explain how weaknesses may limit their effectiveness.
            
            -Generate a natural, fluent description:
                Highlight key strengths that define the player's ability.
                Mention secondary strengths if relevant.
                Point areas for improvement, keeping a constructive tone.
            
            - If data concerns about defense and the player plays in defensive roles (e.g. central back, fullback) use more 100 tokens.
            

        ## Input Format:
            - **Position:** Preferred positions 
            - **Statistics:** Per-match data with **values** and **percentiles** (0-100). A high percentile indicates **strong performance** relative to players in the same position.
        
        ## Output Guidelines:
            - **Professional & concise** language suitable for scouting.
            - **Short** The report MUST be very concise, around 50 tokens, if the position is a defensive role (e.g. Defender, central back, fullback) and data are related to defense use 100 tokens
            - **Plain text only** (no bullet points, code, or structured output).
            - **Do not include raw statistics and percentiles information** (focus on interpretation).
            - **No predictions** about future performance.
            - **Only use provided data**, without speculation.
            - **The description should be role-specific**, considering the player's position.
            - The report regards only one player.
            - The report should be similar to the following example output structure.
            - Don't add new input data
            - [END_REPORT] when you end the description
        
        ## Example Output:
            A well-rounded attacking midfielder with exceptional ability in progressing the ball and creating goal-scoring opportunities. 
            He excels in shot-creating actions, with a strong ability to beat defenders through take-ons. 
            His capacity to contribute directly to goals is elite. 

        ### Input Data:
            - **Position**: {roles_str}
            - **Statistics**: {stat}
            
        ###Generated Report:"""
  
        
        prompts[player_name]['prompt'][k] = prompt

writeJson(prompts, 'Descriptions/player_descriptions.json')

In [ ]:
prompts = {}
records = readJson('Dataset/Fbref/players.json')
records = readJson('Dataset/transfermarkt_fbref_dataset.json')
#records = [x for x in records if x['team'] in ['Roma','Milan','Juventus','Inter','Atalanta','Real Madrid', 'Manchester City']]
#records = [x for x in records if x['team'] in ['Milan']]
records = [x for x in records if 'position' in x.keys() and x['position'] != 'GK']
#records = [x for x in records if x['name'] in ['Alessandro Bastoni', 'Theo Hernandez', 'Benjamin Pavard', 'Fikayo Tomori', 'Francesco Acerbi']]

position_dict = readJson('Dataset/Fbref/position_mapping.json')
#stats_dict = {'Offensive':['Shooting', 'Goal and Shot Creation', 'Possession','Passing', 'Pass Types'], 'Defensive':['Defense', 'Miscellaneous Stats']}
stats_dict = {'Shooting':['Shooting'], 'Passing': ['Goal and Shot Creation','Passing', 'Pass Types'], 'Possession': ['Possession'], 'Defensive':['Defense']}#, 'Miscellaneous Stats']}
new_records = []
for p in records:
    player_name = p['name']
    print(player_name, p['link'], p['position'])
    #prompts[player_name] = {'prompt':{}}
    roles = getRoles(p['position'])
    roles_verb = [position_dict[x.strip()] for x in roles]
    roles_str = ', '.join(roles_verb)
    #stats_dict = p['stats']
    prompts[player_name]['position'] = roles_str
    for k, tab  in list(stats_dict.items())[0:]:
        stat = tab
        stat = {}
        for t in tab:
            stat = stat | p['stats'][t]

        prompt = f""""You are a professional soccer scout with expertise in analyzing players' technical and tactical characteristics.
        ## Task:  
        Generate a very concise description (75 tokens) of a player's {k} performance based on the provided per-match statistics. 
        The analysis should:    
            - Highlight the player's **key strengths** (high percentiles).
            - Identify potential **areas for improvement** (low percentiles).
            - Be **role-specific**, considering the player's position.
        
        ## Reasoning Process:
            - For each statistic, analyze its percentile ranking, classifying performance using these thresholds:
                Excellent (≥90th percentile) -> Major strength
                Very Good (75-89th percentile) -> Significant asset
                Good (50-74th percentile) -> Competent ability
                Average (30-49th percentile) -> Room for improvement
                Weak (<30th percentile) -> Notable weakness

            - Interpret the player's style of play:
                Identify how the player's strengths shape their contributions.
                Explain how weaknesses may limit their effectiveness.
            
            -Generate a natural, fluent description:
                Highlight key strengths that define the player's ability.
                Mention secondary strengths if relevant.
                Point areas for improvement, keeping a constructive tone.
            
            - If data concerns about defense and the player plays in defensive roles (e.g. central back, fullback) use more 100 tokens.
            

        ## Input Format:
            - **Position:** Preferred positions 
            - **Statistics:** Per-match data with **values** and **percentiles** (0-100). A high percentile indicates **strong performance** relative to players in the same position.
        
        ## Output Guidelines:
            - **Professional & concise** language suitable for scouting.
            - **Short** The report MUST be very concise, around 50 tokens, if the position is a defensive role (e.g. Defender, central back, fullback) and data are related to defense use 100 tokens
            - **Plain text only** (no bullet points, code, or structured output).
            - **Do not include raw statistics and percentiles information** (focus on interpretation).
            - **No predictions** about future performance.
            - **Only use provided data**, without speculation.
            - **The description should be role-specific**, considering the player's position.
            - The report regards only one player.
            - The report should be similar to the following example output structure.
            - Don't add new input data
            - [END_REPORT] when you end the description
        
        ## Example Output:
            A well-rounded attacking midfielder with exceptional ability in progressing the ball and creating goal-scoring opportunities. 
            He excels in shot-creating actions, with a strong ability to beat defenders through take-ons. 
            His capacity to contribute directly to goals is elite. 

        ### Input Data:
            - **Position**: {roles_str}
            - **Statistics**: {stat}
            
        ###Generated Report:"""
  
        
        prompts[player_name]['prompt'][k] = prompt

writeJson(prompts, 'Descriptions/prova_stats_v2.json')

Tijjani Reijnders https://fbref.com/en/players/afb61630/scout/12229/Tijjani-Reijnders-Scouting-Report MF (CM-DM)
Christian Pulisic https://fbref.com/en/players/1bf33a9a/scout/12229/Christian-Pulisic-Scouting-Report FW-MF (AM)
Theo Hernandez https://fbref.com/en/players/d4c9725f/scout/12229/Theo-Hernandez-Scouting-Report DF (FB, left)
Rafael Leao https://fbref.com/en/players/20730eae/scout/12229/Rafael-Leao-Scouting-Report FW-MF (AM, left)
Olivier Giroud https://fbref.com/en/players/16ceb862/scout/12229/Olivier-Giroud-Scouting-Report FW
Davide Calabria https://fbref.com/en/players/2146785a/scout/12229/Davide-Calabria-Scouting-Report DF-MF (FB, right)
Ruben Loftus Cheek https://fbref.com/en/players/e97fd090/scout/12229/Ruben-Loftus-Cheek-Scouting-Report MF (AM-WM)
Fikayo Tomori https://fbref.com/en/players/7edfbb8a/scout/12229/Fikayo-Tomori-Scouting-Report DF (CB, left)
Alessandro Florenzi https://fbref.com/en/players/e288d4b3/scout/12229/Alessandro-Florenzi-Scouting-Report DF-FW-MF (AM-

In [157]:
def cleanDesc(desc):
    c_square = desc.count('[END_REPORT]')
    c_plain = desc.count('END_REPORT')
    c = max(c_square, c_plain)
    sep = '[END_REPORT]' if c_square > 0 else 'END_REPORT'
    if c == 0:
        sep = "### Input Data:"
    splits = desc.split(sep)          
    splits = [s.strip() for s in splits]
    if splits[0] == '':
        return splits[1]
    else:
        return splits[0]


def getTeamPlayerIds(dataset):
    teams = set(x['team'] for x in dataset)
    dataset = [x for x in dataset if 'position' in x.keys() and x['position'] != 'GK']
    team_player_list = {}
    for t in teams:
        players = []
        for p in dataset:
            if p['team'] == t:
                players.append(p['id'])
        
        team_player_list[t] = players
    
    return team_player_list


def isDefender(p):
    if 'Defender' in p['position_str'] or 'back'in p['position_str']:
        return True
    return False


Prompt per generare profilo delle squadre

In [ ]:
player_desc = readJson("Descriptions/Descriptions/player_descriptions.json")
dataset_in = readJson("Dataset/transfermarkt_fbref_dataset.json")
team_player_list = getTeamPlayerIds(dataset_in)
team_dict = {}
for team, ps in team_player_list.items():
    team_dict[team] = {}
    team_dict[team]['prompt'] = {}

    prompt=f"""You are a professional soccer analyst specializing in squad-building strategies.
        ## Task:
        Analyze the individual descriptions of a team's players and determine key attributes or tactical elements that are missing from the squad, which could impact overall balance and effectiveness.
        ## Reasoning process:
        1. Group players by position (Defenders, Midfielders, Forwarders) based on the provided descriptions.
        2. Identify the prioritized characteristics by analyzing the most frequently mentioned strengths and weaknesses for each role.
        3. Analyze potential gaps in squad composition by detecting missing traits that could improve balance (e.g., lack of creativity in midfield, absence of aerially dominant forwards, limited pace in defense).
        4. Summarize the findings, highlighting both the team's key characteristics and the notable absences that could impact performance.
        
        ## Input format:
        Team Name: [Team Name]
        Player Descriptions: A list of individual player descriptions, each one providing insights into the player's strengths and weaknesses.
        
        ## Output guidelines:
        Provide a concise (200 tokens) summary of the characteristics valued by the team for each role.
        Ensure the descriptions are coherent and reflect a structured playing philosophy.
        Use professional and analytical language.
        Do not include individual player names—focus on general trends.
        Plain text only (no bullet points, code, or structured output).
        Use [END_REPORT] when you end the description
        The report must have the same structure of the example output
        
        ## Example Input:
        Team Name: FC Example
        Player Descriptions:
        - Defender, Central back: A physical and aggressive center-back who dominates in aerial duels and defensive tackles but struggles in ball progression.
        - Fullback: A dynamic fullback with high stamina and strong defensive positioning, yet limited attacking contributions.
        - Midfielder, Central midfielder: A central midfielder with outstanding pressing ability and quick passing, but lacking goal-scoring instinct.
        - Attacking midfielder: An attacking winger with elite dribbling and acceleration, excelling in 1v1 situations but offering little defensive work.
        - Forwarder: A striker with a clinical finishing ability and strong positioning inside the box, but not very involved in buildup play.
        ## Example Output:
        FC Example builds its squad around a physically dominant defensive structure, center-backs with strong aerial ability and defensive aggression, though they contribute less in possession. Fullbacks are selected for their defensive stability rather than attacking impact. In midfield, the club prioritizes high-intensity pressing and quick ball circulation over goal-scoring ability. Their attacking philosophy emphasizes pace and individual skill, with wingers excelling in 1v1 situations and strikers focused on efficient finishing rather than playmaking.
        
        ## Input data:
        Team Name: {team}
        Player descriptions:
        """
    #ps = list(players.keys())

        
    for p in ps:
        desc_keys = list(player_desc[p]['description'].keys())[:-1]
        if isDefender(player_desc[p]):
            desc_keys = desc_keys[::-1]
            
        desc = ''.join([player_desc[p]['description'][k] for k in desc_keys])
        #prompt+= f"- {p}: {cleanDesc(players[p]['description']['summary'])}\n"
        prompt+= f"- {player_desc[p]['position_str']}: {desc}\n"
        #prompt+= f"- {players[p]['position']}: {desc}\n"

    prompt+="###Generated Report:"
    #missing_prompt+="###Generated Report:"

    team_dict[team]['prompt']['general'] = prompt
    #team_dict[team]['prompt']['missing'] = missing_prompt


writeJson(team_dict, 'Descriptions/team_descriptions.json')

Tijjani Reijnders
Christian Pulisic
Theo Hernandez
Rafael Leao
Olivier Giroud
Davide Calabria
Ruben Loftus Cheek
Fikayo Tomori
Alessandro Florenzi
Malick Thiaw
Yacine Adli
Matteo Gabbia
Simon Kjaer
Yunus Musah
Ismael Bennacer
Samuel Chukwueze
Luka Jovic
Rade Krunic
Noah Okafor


In [160]:
dataset = readJson('Dataset/transfermarkt_fbref_dataset.json')
dataset_stat_null = [x for x in dataset if x['stats'] == {}]
len(dataset_stat_null)

353